In [68]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_curve, auc
from peft import LoraConfig, get_peft_model  # For efficient fine-tuning



In [69]:
df = pd.read_excel('bioactivity_dataset_cleaned_outliers.xlsx')
df['mol'] = df['canonical_smiles'].apply(Chem.MolFromSmiles)
df = df.dropna(subset=['mol'])

# Compute/scale descriptors (if not already)
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros(4)
    return np.array([Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
                     rdMolDescriptors.CalcNumHBD(mol), rdMolDescriptors.CalcNumHBA(mol)])

df[['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']] = df['canonical_smiles'].apply(compute_descriptors).tolist()

scaler = StandardScaler()
desc_cols = ['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']
df[desc_cols] = scaler.fit_transform(df[desc_cols])

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667 entries, 0 to 5666
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   canonical_smiles   5667 non-null   object 
 1   MW                 5667 non-null   float64
 2   LogP               5667 non-null   float64
 3   NumHDonors         5667 non-null   float64
 4   NumHAcceptors      5667 non-null   float64
 5   pIC50              5667 non-null   float64
 6   bioactivity_class  5667 non-null   object 
 7   bioactivity        5667 non-null   int64  
 8   mol                5667 non-null   object 
dtypes: float64(5), int64(1), object(3)
memory usage: 398.6+ KB


In [71]:
df.head(3)

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class,bioactivity,mol
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,0.524109,0.241522,1.32487,0.282913,8.6,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001EB619...
1,O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...,0.732961,0.621072,1.32487,0.282913,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001EB619...
2,O=C(/C=C/c1cccc(C(C(=O)Nc2ccccc2)C(=O)Nc2ccccc...,0.036571,-0.025903,1.32487,-0.713216,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001EAC98...


In [72]:
df.drop(columns=['pIC50', 'bioactivity_class'], inplace=True)

In [73]:
df.iloc[0]['canonical_smiles']

'O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc12)NO'

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667 entries, 0 to 5666
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   canonical_smiles  5667 non-null   object 
 1   MW                5667 non-null   float64
 2   LogP              5667 non-null   float64
 3   NumHDonors        5667 non-null   float64
 4   NumHAcceptors     5667 non-null   float64
 5   bioactivity       5667 non-null   int64  
 6   mol               5667 non-null   object 
dtypes: float64(4), int64(1), object(2)
memory usage: 310.0+ KB


In [75]:
df['bioactivity'].value_counts()

bioactivity
1    4641
0    1026
Name: count, dtype: int64

In [97]:
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTENC


train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['bioactivity'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

X = train_df[['canonical_smiles'] + desc_cols]
y = train_df['bioactivity']

smote_tomek = SMOTENC(random_state=42, categorical_features=[0])
desc_X = train_df[desc_cols].values
smiles_X = train_df['canonical_smiles'].values.reshape(-1, 1)

# Concatenate SMILES and descriptors for SMOTENC
X_concat = np.concatenate([smiles_X, desc_X], axis=1)

X_resampled, y_resampled = smote_tomek.fit_resample(X_concat, y)

smiles_res = X_resampled[:, 0]
desc_res = X_resampled[:, 1:].astype(float)
train_df_resampled = pd.DataFrame(desc_res, columns=desc_cols)
train_df_resampled['canonical_smiles'] = smiles_res
train_df_resampled['bioactivity'] = y_resampled

train_df_resampled = train_df_resampled[['canonical_smiles'] + desc_cols + ['bioactivity']]

# Reconstruct balanced train_df

tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')

class BioactivityDataset(Dataset):
    def __init__(self, df, tokenizer, desc_scaler):
        self.smiles = df['canonical_smiles'].tolist()
        self.labels = df['bioactivity'].tolist()
        self.descs = df[desc_cols].values
        #self.desc_scaler = desc_scaler
        self.encodings = tokenizer(self.smiles, truncation=True, padding=True, max_length=512, return_tensors='pt')
    
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        item['descriptors'] = torch.tensor(self.descs[idx], dtype=torch.float)
        return item

In [77]:
class MoleculeDataset(Dataset):
    def __init__(self, df, tokenizer, num_features, scaler=None, is_train=False, max_length=128):
        self.smiles = df['canonical_smiles'].values
        self.num_features = df[num_features].values.astype(np.float32)
        self.labels = df['bioactivity'].values.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length
        # if is_train:
        #     self.scaler = StandardScaler().fit(self.num_features)
        # else:
        #     self.scaler = scaler
        #self.num_features = self.scaler.transform(self.num_features)

    def __len__(self):
        return len(self.smiles)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(self.smiles[idx], truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        num_feats = torch.tensor(self.num_features[idx])
        label = torch.tensor(self.labels[idx])
        return input_ids, attention_mask, num_feats, label

In [50]:
train_dataset = MoleculeDataset(train_df, tokenizer, ['MW', 'LogP', 'NumHDonors', 'NumHAcceptors'], scaler)
val_dataset = MoleculeDataset(val_df, tokenizer, ['MW', 'LogP', 'NumHDonors', 'NumHAcceptors'], scaler)
test_dataset = MoleculeDataset(test_df, tokenizer, ['MW', 'LogP', 'NumHDonors', 'NumHAcceptors'], scaler)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [98]:
# Augmentation (3 variants for ~15k samples)
def augment_smiles(smiles, num_aug=3):
    mol = Chem.MolFromSmiles(smiles)
    variants = []
    descriptors = []
    if mol is None:
        variants.append(smiles)
        descriptors.append(compute_descriptors(smiles))
        return variants, descriptors
    variants.append(Chem.MolToSmiles(mol, canonical=True))
    descriptors.append(compute_descriptors(variants[0]))
    for _ in range(num_aug):
        try:
            variant = Chem.MolToSmiles(mol, canonical=False, doRandom=True)
            if Chem.MolFromSmiles(variant) is not None and variant not in variants:
                variants.append(variant)
                descriptors.append(compute_descriptors(variant))
        except Exception:
            pass
    return variants, descriptors

aug_df = []
for _, row in train_df.iterrows():
    for var_smiles, desc in zip(*augment_smiles(row['canonical_smiles'])):
        aug_row = row.copy()
        aug_row['canonical_smiles'] = var_smiles
        aug_row[desc_cols] = desc
        aug_df.append(aug_row)
    
train_df = pd.DataFrame(aug_df)

In [99]:
train_df['bioactivity'].value_counts()

bioactivity
1    14831
0     3277
Name: count, dtype: int64

In [100]:
train_df.duplicated().sum()

0

In [101]:
train_dataset = BioactivityDataset(train_df, tokenizer, scaler)
train_dataset_resampled = BioactivityDataset(train_df_resampled, tokenizer, scaler)
val_dataset = BioactivityDataset(val_df, tokenizer, scaler)
test_dataset = BioactivityDataset(test_df, tokenizer, scaler)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
train_loader_resampled = DataLoader(train_dataset_resampled, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [67]:
def augment_smiles(smiles, num_aug=3):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [smiles]
    variants = [Chem.MolToSmiles(mol, canonical=True)]
    for _ in range(num_aug):
        try:
            variant = Chem.MolToSmiles(mol, canonical=False, doRandom=True)
            # Only add if valid and not a duplicate
            if Chem.MolFromSmiles(variant) is not None and variant not in variants:
                variants.append(variant)
        except Exception:
            pass
    return variants

augmented = augment_smiles('O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc12)NO', num_aug=5)
print(augmented)

['O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc12)NO', 'c1(ccccc1)COC(NC(C(Nc1c2c(cccn2)ccc1)=O)CCCCCC(=O)NO)=O', 'c1cc2cccc(c2nc1)NC(=O)C(CCCCCC(=O)NO)NC(OCc1ccccc1)=O', 'c12c(cccc2cccn1)NC(=O)C(NC(=O)OCc1ccccc1)CCCCCC(=O)NO', 'c12c(nccc1)c(NC(C(NC(OCc1ccccc1)=O)CCCCCC(NO)=O)=O)ccc2', 'O=C(CCCCCC(C(Nc1cccc2c1nccc2)=O)NC(=O)OCc1ccccc1)NO']


In [72]:
model_name = "seyonec/ChemBERTa-zinc-base-v1"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
bert       = AutoModel.from_pretrained(model_name)   # <-- only the encoder
# -------------------------------------------------

def encode_batch(smiles_list, num_feats_tensor):
    # 1. Tokenise
    enc = tokenizer(
        smiles_list,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors='pt'
    )
    input_ids      = enc['input_ids']          # [B, 128]
    attention_mask = enc['attention_mask']     # [B, 128]

    # 2. BERT forward (no gradient)
    with torch.no_grad():
        out = bert(input_ids=input_ids, attention_mask=attention_mask)
        bert_vec = out.pooler_output           # [B, 768]  <--- DENSE!

    # 3. Concatenate numeric descriptors
    final_vec = torch.cat([bert_vec, num_feats_tensor], dim=1)  # [B, 772]
    return final_vec

In [73]:
vec=encode_batch(['CCO', 'CCN'], torch.tensor([[0.1, 0.2, 0.3, 0.4],[0.5, 0.6, 0.7, 0.8]]))


In [74]:
vec.shape

torch.Size([2, 772])

In [102]:
train_df['bioactivity'].value_counts(), val_df['bioactivity'].value_counts(), test_df['bioactivity'].value_counts()

(bioactivity
 1    14831
 0     3277
 Name: count, dtype: int64,
 bioactivity
 1    454
 0    113
 Name: count, dtype: int64,
 bioactivity
 1    475
 0     92
 Name: count, dtype: int64)

In [103]:
len(train_loader_resampled.dataset), len(train_loader.dataset)

(7424, 18108)

In [104]:
class FineTunedMultimodalModel(nn.Module):
    def __init__(self, num_desc=4, emb_dim=768, hidden_dim=256, dropout=0.4, num_layers_to_unfreeze=2):
        super().__init__()
        self.smiles_encoder = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')

        self.config = self.smiles_encoder.config
        self.config.num_labels = 1  # Optional: Set for classification task

        # Freeze all except last num_layers_to_unfreeze
        for param in self.smiles_encoder.parameters():
            param.requires_grad = False
        unfreeze_modules = self.smiles_encoder.encoder.layer[-num_layers_to_unfreeze:]
        for module in unfreeze_modules:
            for param in module.parameters():
                param.requires_grad = True
        
        self.desc_proj = nn.Linear(num_desc, emb_dim)
        self.fusion = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, input_ids=None, attention_mask=None, descriptors=None, **kwargs):
        outputs = self.smiles_encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        smiles_emb = outputs.last_hidden_state[:, 0, :]
        desc_emb = self.desc_proj(descriptors)
        fused = torch.cat([smiles_emb, desc_emb], dim=-1)
        return self.fusion(fused).squeeze(-1)

model = FineTunedMultimodalModel(num_layers_to_unfreeze=1)  # Start with 2 for small data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

FineTunedMultimodalModel(
  (smiles_encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(767, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              

In [ ]:
class ChemBERTaClassifier(nn.Module):
    def __init__(self, model_name, num_num_feats=4, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.fc_concat = nn.Linear(self.bert.config.hidden_size + num_num_feats, 128)
        self.fc_out = nn.Linear(128, 1)

    def forward(self, input_ids, attention_mask, num_feats):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # [CLS] token representation
        concat = torch.cat([pooled, num_feats], dim=1)
        x = self.dropout(torch.relu(self.fc_concat(concat)))
        return torch.sigmoid(self.fc_out(x)).squeeze()

In [59]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig( task_type=TaskType.SEQ_CLS, r=4, lora_alpha=8, target_modules=["query", "value"], lora_dropout=0.1)

In [60]:
model = get_peft_model(model, lora_config)
 
# Print trainable params (should be <1% of total)
model.print_trainable_parameters()  

trainable params: 73,728 || all params: 44,608,001 || trainable%: 0.1653


In [105]:
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(train_df['bioactivity']), y=train_df['bioactivity'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)  # Low LR for fine-tuning
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

criterion = BCEWithLogitsLoss(pos_weight=class_weights[1])
epochs = 50  # Longer for fine-tuning
best_auc = 0
patience, counter = 5, 0

In [106]:
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        descriptors = batch['descriptors'].to(device)
        labels = batch['labels'].float().to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask, descriptors=descriptors)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    
    # Validation
    model.eval()
    val_preds, val_labels = [], []
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            descriptors = batch['descriptors'].to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attention_mask, descriptors)
            probs = torch.sigmoid(logits).cpu().numpy()
            val_preds.extend(probs)
            val_labels.extend(labels.cpu().numpy())
            loss = criterion(logits, labels.float())  # Compute loss for this batch
            val_loss += loss.item()  #
    
    scheduler.step(val_loss / len(val_loader))  # Adjust LR
    
    auc = roc_auc_score(val_labels, val_preds)
    print(f'Epoch {epoch+1}: Train Loss {train_loss/len(train_loader):.4f}, Val AUROC {auc:.4f}')
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'best_finetuned_model_.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

Epoch 1: Train Loss 0.4790, Val AUROC 0.6983
Epoch 2: Train Loss 0.4236, Val AUROC 0.7070
Epoch 3: Train Loss 0.4036, Val AUROC 0.7065
Epoch 4: Train Loss 0.3968, Val AUROC 0.7054
Epoch 5: Train Loss 0.3953, Val AUROC 0.7051
Epoch 6: Train Loss 0.3954, Val AUROC 0.7063
Epoch 7: Train Loss 0.3952, Val AUROC 0.7059
Early stopping!


In [107]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Run inference on test set
model.eval()
test_preds = []
test_labels = []
test_probs = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = model(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_probs.extend(probs)
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
auc = roc_auc_score(test_labels, test_probs)
print(f"AUC: {auc}")
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))

AUC: 0.7282723112128148
Accuracy: 0.5
accuracy another:  0.8377425044091711


In [108]:
print(classification_report(test_labels, test_preds))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        92
           1       0.84      1.00      0.91       475

    accuracy                           0.84       567
   macro avg       0.42      0.50      0.46       567
weighted avg       0.70      0.84      0.76       567



e:\ML\BioActivity\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\ML\BioActivity\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\ML\BioActivity\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [21]:
# Recreate the model architecture
model_1 = FineTunedMultimodalModel(num_layers_to_unfreeze=1)
model_1.load_state_dict(torch.load('best_finetuned_model.pth', map_location='cpu'))  # or 'cuda' if using GPU
model_1.to(device)
model_1.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FineTunedMultimodalModel(
  (smiles_encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(600, 384, padding_idx=1)
      (position_embeddings): Embedding(515, 384, padding_idx=1)
      (token_type_embeddings): Embedding(1, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.144, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-2): 3 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.109, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
          

In [22]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Run inference on test set

test_preds = []
test_labels = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = model_1(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))
roc_auc = roc_auc_score(test_labels, test_preds)
print(f"ROC AUC: {roc_auc}")

Accuracy: 0.9560995049920296
accuracy another:  0.9730921923246582
ROC AUC: 0.9560995049920296


In [54]:
class ChemBERTaClassifier(nn.Module):
    def __init__(self, model_name, num_num_feats=4, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.fc_concat = nn.Linear(self.bert.config.hidden_size + num_num_feats, 128)
        self.fc_out = nn.Linear(128, 1)

    def forward(self, input_ids, attention_mask, num_feats):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # [CLS] token representation
        concat = torch.cat([pooled, num_feats], dim=1)
        x = self.dropout(torch.relu(self.fc_concat(concat)))
        return torch.sigmoid(self.fc_out(x)).squeeze()

In [59]:
import torch.optim as optim
# Step 3: Training and evaluation
def train_and_evaluate(train_df, val_df, test_df):
    model_name = "DeepChem/ChemBERTa-77M-MLM"  # Lightweight ChemBERTa
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    num_features = ['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']
    
    # Datasets
    train_dataset = MoleculeDataset(train_df, tokenizer, num_features, is_train=True)
    val_dataset = MoleculeDataset(val_df, tokenizer, num_features)
    test_dataset = MoleculeDataset(test_df, tokenizer, num_features)
    
    # Dataloaders (no collate needed, as padding is handled in tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Smaller batch for BERT
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    # Model, loss (weighted for imbalance), optimizer
    model = ChemBERTaClassifier(model_name)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    # Positive weight for minority class (inactive:0); from train counts ~3712 active / 821 inactive ≈ 4.5
    pos_weight = torch.tensor([3712 / 821]).to(device)  # Penalize misclassifying inactive
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=2e-5)  # Low LR for fine-tuning
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    
    # Training loop
    epochs = 50  # Fewer epochs as fine-tuning converges faster
    best_val_acc = 0
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for input_ids, attention_mask, num_feats, labels in train_loader:
            input_ids, attention_mask, num_feats, labels = input_ids.to(device), attention_mask.to(device), num_feats.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask, num_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        print(f'Epoch {epoch+1}: Train Loss = {train_loss / len(train_loader):.4f}')
        
        # Validation
        model.eval()
        val_preds, val_true = [], []
        val_loss = 0
        with torch.no_grad():
            for input_ids, attention_mask, num_feats, labels in val_loader:
                input_ids, attention_mask, num_feats, labels = input_ids.to(device), attention_mask.to(device), num_feats.to(device), labels.to(device)
                outputs = model(input_ids, attention_mask, num_feats)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                val_preds.extend((outputs > 0.5).cpu().numpy())
                val_true.extend(labels.cpu().numpy())
        val_acc = accuracy_score(val_true, val_preds)
        scheduler.step(val_loss / len(val_loader))
        print(f'Val Loss = {val_loss / len(val_loader):.4f}, Val Acc = {val_acc:.4f}')
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_chemberta.pth')
    
    # Test evaluation
    model.load_state_dict(torch.load('best_chemberta.pth'))
    model.eval()
    test_preds, test_true = [], []
    with torch.no_grad():
        for input_ids, attention_mask, num_feats, labels in test_loader:
            input_ids, attention_mask, num_feats, labels = input_ids.to(device), attention_mask.to(device), num_feats.to(device), labels.to(device)
            outputs = model(input_ids, attention_mask, num_feats)
            test_preds.extend((outputs > 0.5).cpu().numpy())
            test_true.extend(labels.cpu().numpy())
    test_acc = accuracy_score(test_true, test_preds)
    test_f1 = f1_score(test_true, test_preds)
    print(f'Test Accuracy: {test_acc:.4f}, Test F1: {test_f1:.4f}')

# Usage: Replace with your DataFrames
# train_and_evaluate(train_df, val_df, test_df)

In [60]:
train_and_evaluate(train_df, val_df, test_df)

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MLM and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1: Train Loss = 1.7830
Val Loss = 1.5265, Val Acc = 0.8007
Epoch 2: Train Loss = 1.4602
Val Loss = 1.4144, Val Acc = 0.8007
Epoch 3: Train Loss = 1.4125
Val Loss = 1.4022, Val Acc = 0.8007
Epoch 4: Train Loss = 1.4042
Val Loss = 1.3989, Val Acc = 0.8007
Epoch 5: Train Loss = 1.4012
Val Loss = 1.3977, Val Acc = 0.8007
Epoch 6: Train Loss = 1.3999
Val Loss = 1.3971, Val Acc = 0.8007
Epoch 7: Train Loss = 1.3993
Val Loss = 1.3967, Val Acc = 0.8007
Epoch 8: Train Loss = 1.3989
Val Loss = 1.3965, Val Acc = 0.8007
Epoch 9: Train Loss = 1.3986
Val Loss = 1.3964, Val Acc = 0.8007
Epoch 10: Train Loss = 1.3984
Val Loss = 1.3963, Val Acc = 0.8007
Epoch 11: Train Loss = 1.3983
Val Loss = 1.3962, Val Acc = 0.8007
Epoch 12: Train Loss = 1.3982
Val Loss = 1.3961, Val Acc = 0.8007
Epoch 13: Train Loss = 1.3981
Val Loss = 1.3961, Val Acc = 0.8007
Epoch 14: Train Loss = 1.3980
Val Loss = 1.3961, Val Acc = 0.8007
Epoch 15: Train Loss = 1.3980
Val Loss = 1.3960, Val Acc = 0.8007
Epoch 16: Train Los